In [0]:
from pyspark import pipelines as dp

from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    StringType,
    StructField,
    StructType,
)

In [0]:
# Unity Catalog external volume containing the untouched EIA API responses.
LANDING_ROOT = (
    "/Volumes/dbw_energy_analytics_dev/"
    "raw/raw_files"
)

REGION_PATH = f"{LANDING_ROOT}/region-data/"
FUEL_PATH = f"{LANDING_ROOT}/fuel-type-data/"

In [0]:
# Endpoint-specific schemas for response.data.
# Keep source measurements as strings in Bronze. Type conversion and
# business validation belong in Silver.

region_row_schema = StructType([
    StructField("period", StringType(), True),
    StructField("respondent", StringType(), True),
    StructField("respondent-name", StringType(), True),
    StructField("type", StringType(), True),
    StructField("type-name", StringType(), True),
    StructField("value", StringType(), True),
    StructField("value-units", StringType(), True),
])

fuel_row_schema = StructType([
    StructField("period", StringType(), True),
    StructField("respondent", StringType(), True),
    StructField("respondent-name", StringType(), True),
    StructField("fueltype", StringType(), True),
    StructField("type-name", StringType(), True),
    StructField("value", StringType(), True),
    StructField("value-units", StringType(), True),
])

In [0]:
# Complete top-level EIA response schema.
#
# The top level of an EIA response contains: warnings (only on some responses),
# response, request, apiVersion and ExcelAddInVersion.
#
# `warnings` and `request` are declared as StringType on purpose: Spark keeps
# any JSON object or list as raw text in a StringType column. We never use them
# (the original request stays in the raw files), and this way a new request
# parameter or warning from EIA does not fill _rescued_data on every row.
#
# The rescued data column (_rescued_data) is added by the rescuedDataColumn
# option in explode_landing, so it is not declared here.
#
# The response.data record schema differs by endpoint.

def response_schema(row_schema):
    return StructType([
        StructField("ExcelAddInVersion", StringType(), True),
        StructField("apiVersion", StringType(), True),
        StructField("warnings", StringType(), True),
        StructField("request", StringType(), True),
        StructField(
            "response",
            StructType([
                StructField("total", StringType(), True),
                StructField("dateFormat", StringType(), True),
                StructField("frequency", StringType(), True),
                StructField("data", ArrayType(row_schema), True),
                StructField("description", StringType(), True),
            ]),
            True,
        ),
    ])


REGION_SCHEMA = response_schema(region_row_schema)
FUEL_SCHEMA = response_schema(fuel_row_schema)

In [0]:
# Pull metadata is encoded in the landing filename:
# YYYYMMDDTHHMMSSZ_<pull_mode>_<RESPONDENT>_wNNN.json
#
# pulled_at is the ingestion/retrieval timestamp, not the measurement period.
#
# The "X" in the timestamp format reads the trailing Z as a UTC offset, so the
# result is the same instant whatever the session time zone is.
#
# If a file name does not match the pattern (for example a stray file), both
# columns are null. The guard matters: to_timestamp on an empty string raises
# an error in Spark's strict (ANSI) mode and would fail the whole pipeline update.

FILE_NAME = F.col("_metadata.file_name")

_STAMP = F.regexp_extract(
    FILE_NAME,
    r"^(\d{8}T\d{6}Z)_",
    1,
)

_MODE = F.regexp_extract(
    FILE_NAME,
    r"^\d{8}T\d{6}Z_([a-z]+)_",
    1,
)

PULLED_AT = F.when(
    _STAMP != "",
    F.to_timestamp(
        _STAMP,
        "yyyyMMdd'T'HHmmssX",
    ),
).alias("pulled_at")

PULL_MODE = F.when(
    _MODE != "",
    _MODE,
).alias("pull_mode")

SOURCE_FILE = F.col("_metadata.file_path").alias("source_file")

In [0]:
# Shared Auto Loader reader and flattening logic.
#
# Auto Loader discovers files under each endpoint directory, including
# ingest_date partition subdirectories. The pipeline manages checkpoint
# and schema-tracking state.
#
# Step 1: explode_outer() creates one output row per record in response.data.
# explode_outer (not explode) is used on purpose: a response whose data list is
# empty, or whose structure changed, still produces one row of nulls, so it shows
# up in the checks and in _rescued_data instead of disappearing silently.
#
# Step 2: record.* picks the columns, and the hyphenated source names are
# renamed to underscores so queries do not need backticks.

RENAME_COLUMNS = {
    "respondent-name": "respondent_name",
    "type-name": "type_name",
    "value-units": "value_units",
}


def explode_landing(endpoint_path, schema):
    raw_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("multiLine", "true")
        .option("rescuedDataColumn", "_rescued_data")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .schema(schema)
        .load(endpoint_path)
    )

    exploded_df = raw_df.select(
        F.explode_outer(F.col("response.data")).alias("record"),
        PULLED_AT,
        PULL_MODE,
        SOURCE_FILE,
        F.col("_rescued_data"),
    )

    return exploded_df.select(
        "record.*",
        "pulled_at",
        "pull_mode",
        "source_file",
        "_rescued_data",
    ).withColumnsRenamed(RENAME_COLUMNS)

In [0]:
# Warn-only data quality expectations.
# These expectations report invalid rows but do not drop them or fail
# the pipeline update.

CHECKS_REGION = {
    "valid_period": "period IS NOT NULL AND period <> ''",
    "valid_respondent": "respondent IS NOT NULL AND respondent <> ''",
    "valid_type": "type IS NOT NULL AND type <> ''",
    "valid_value": "value IS NOT NULL AND value <> ''",
    "valid_pulled_at": "pulled_at IS NOT NULL",
    "valid_pull_mode": "pull_mode IN ('initial', 'landing')",
    "no_rescued_data": "_rescued_data IS NULL",
}

CHECKS_FUEL = {
    "valid_period": "period IS NOT NULL AND period <> ''",
    "valid_respondent": "respondent IS NOT NULL AND respondent <> ''",
    "valid_fueltype": "fueltype IS NOT NULL AND fueltype <> ''",
    "valid_value": "value IS NOT NULL AND value <> ''",
    "valid_pulled_at": "pulled_at IS NOT NULL",
    "valid_pull_mode": "pull_mode IN ('initial', 'landing')",
    "no_rescued_data": "_rescued_data IS NULL",
}

In [0]:
@dp.table(
    name="bronze_region_data",
    comment=(
        "Bronze records from the EIA region-data endpoint: demand, "
        "day-ahead demand forecast, net generation and total interchange. "
        "Source values are retained as strings with ingestion lineage."
    ),
)
@dp.expect_all(CHECKS_REGION)
def bronze_region_data():
    return explode_landing(
        REGION_PATH,
        REGION_SCHEMA,
    )

In [0]:
@dp.table(
    name="bronze_fuel_type_data",
    comment=(
        "Bronze records from the EIA fuel-type-data endpoint. "
        "Hourly generation by fuel type is retained as source strings "
        "with ingestion lineage."
    ),
)
@dp.expect_all(CHECKS_FUEL)
def bronze_fuel_type_data():
    return explode_landing(
        FUEL_PATH,
        FUEL_SCHEMA,
    )